In [1]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 6.5 MB/s eta 0:00:00a 0:00:01
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Cloning into 'detectron2'...
remote: Enumerating objects: 15943, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 15943 (delta 5), reused 4 (delta 4), pack-reused 15931 (from 3)
Receiving objects: 100% (15943/15943), 6.71 MiB | 24.19 MiB/s, done.
Resolving deltas: 100% (11334/11334), done.
Ignoring dataclasses: markers 'python_version < "3.7"' don't match you

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as funcy
from detectron2.modeling import BACKBONE_REGISTRY, META_ARCH_REGISTRY, Backbone, ShapeSpec, GeneralizedRCNN
from detectron2.layers import Conv2d, ShapeSpec, get_norm, ROIAlign
from detectron2.modeling.roi_heads import StandardROIHeads
from detectron2.modeling.proposal_generator import RPN
from detectron2.modeling import build_backbone,build_proposal_generator, build_roi_heads
from detectron2.modeling.backbone.fpn import FPN
import numpy as np

In [3]:
class Att(nn.Module):
    def __init__(self,chl,redu=16):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(chl,chl//redu,bias=False),
                             nn.ReLU(inplace=True),
                             nn.Linear(chl//redu,chl,bias=False))
        self.spa=nn.Conv2d(2,1,kernel_size=7,padding=3,bias=False)
    def forward(self,x):
        b,c,h,w=x.size()
        avgg=self.fc(funcy.adaptive_avg_pool2d(x,1).view(b,c))
        maxx=self.fc(funcy.adaptive_max_pool2d(x,1).view(b,c))
        ou=torch.sigmoid(avgg+maxx).view(b,c,1,1)
        x=x*ou
        avgg=torch.mean(x,dim=1,keepdim=True)
        maxx,_=torch.max(x,dim=1,keepdim=True)
        spat=torch.sigmoid(self.spa(torch.cat([avgg,maxx],dim=1)))
        return x*spat

In [4]:
@BACKBONE_REGISTRY.register()
class LiHR01(Backbone):
    def __init__(self,cfg,input_shape):
        super().__init__()
        self.b1=nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1),
                             nn.BatchNorm2d(32),
                             nn.ReLU(),
                             nn.Conv2d(32,32,3,stride=2,padding=1),
                             nn.BatchNorm2d(32),
                             nn.ReLU())
        self.b2=nn.Sequential(nn.Conv2d(32,64,3,stride=2,padding=1),
                             nn.BatchNorm2d(64),
                             nn.ReLU())
        self.b3=nn.Sequential(nn.Conv2d(64,128,3,stride=2,padding=1),
                             nn.BatchNorm2d(128),
                             nn.ReLU())
        self.b4=nn.Sequential(nn.Conv2d(128,256,3,stride=2,padding=1),
                             nn.BatchNorm2d(256),
                             nn.ReLU())
        self.outf=['p2','p3','p4','p5']
        self.ofc={"p2": 32, "p3": 64, "p4": 128, "p5": 256}
        self.ofs={"p2": 4, "p3": 8, "p4": 16, "p5": 32}
    def forward(self,x):
        ou={}
        x1=self.b1(x)
        ou['p2']=x1
        x2=self.b2(x1)
        ou['p3']=x2
        x3=self.b3(x2)
        ou['p4']=x3
        x4=self.b4(x3)
        ou['p5']=x4
        return ou
    def output_shape(self):
        return {name: ShapeSpec(channels=self.ofc[name], stride=self.ofs[name]) 
                for name in self.outf}

In [5]:
class Depthead(nn.Module):
    def __init__(self,feat):
        super().__init__()
        self.dec=nn.Sequential(
            nn.Conv2d(256,128,3,padding=1),nn.ReLU(),
            nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True),
            nn.Conv2d(128,64,3,padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True),
            nn.Conv2d(64,32,3,padding=1),nn.ReLU(),
            nn.Upsample(scale_factor=4,mode='bilinear',align_corners=True),
            nn.Conv2d(32,1,3,padding=1),
            nn.Sigmoid()
        )
    def forward(self,fpnn):
        return self.dec(fpnn['p5'])

In [6]:
class TTCHead(nn.Module):
    def __init__(self, inn=256):
        super().__init__()
        self.lin=nn.Sequential(
            nn.Linear(inn*2,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )
    def forward(self,cf,pf):
        c=funcy.adaptive_avg_pool2d(cf,1).view(cf.size(0),-1)
        p=funcy.adaptive_avg_pool2d(pf,1).view(pf.size(0),-1)
        return self.lin(torch.cat([c,p],dim=1))

In [7]:
@META_ARCH_REGISTRY.register()
class TereNaina(GeneralizedRCNN):
    def __init__(self,cfg):
        nn.Module.__init__(self)
        self.device=torch.device(cfg.MODEL.DEVICE)
        rawkamar=build_backbone(cfg)
        self.kamar=FPN(bottom_up=rawkamar,
                      in_features=['p2','p3','p4','p5'],out_channels=256,
                      norm='',top_block=None)
        self.proposal_generator=build_proposal_generator(cfg,self.kamar.output_shape())
        self.roi_heads = build_roi_heads(cfg, self.kamar.output_shape())
        self.att=nn.ModuleDict({k:Att(256) for k in ["p2", "p3", "p4", "p5"]})
        self.deep=Depthead(256)
        self.ttc=TTCHead(256)
        self.input_format = cfg.INPUT.FORMAT
        self.pixel_mean = torch.Tensor(cfg.MODEL.PIXEL_MEAN).view(-1, 1, 1)
        self.pixel_std = torch.Tensor(cfg.MODEL.PIXEL_STD).view(-1, 1, 1)
        self.to(torch.device(cfg.MODEL.DEVICE))
    def forward(self,inn):
        if not self.training:return self.inference(inn)
        img=self.preprocess_image(inn)
        feat=self.kamar(img.tensor)
        feat = {k: self.att[k](v) for k, v in feat.items()}
        prop,lprop=self.proposal_generator(img,feat,inn)
        _,ldet=self.roi_heads(img,feat,prop,inn)
        deppre=self.deep(feat)
        lttc=torch.tensor(0.0).to(self.device)
        if 'prev_img' in inn[0]:
            previmg=torch.stack([x['prev_img'] for x in inn]).to(self.device)
            previmg=(previmg-self.pixel_mean)/self.pixel_std
            with torch.no_grad():
                pfeat=self.kamar(previmg)
                pfeat={k: self.att[k](v) for k, v in pfeat.items()}
                pdepth=self.deep(pfeat)
            ttc_pred = self.ttc(feat['p5'], pfeat['p5'])
            z_curr = torch.mean(deppre, dim=[1, 2, 3])
            z_prev = torch.mean(pdepth, dim=[1, 2, 3])
            velocity = (z_prev - z_curr).abs().clamp(min=1e-3)
            gt_ttc = (z_curr / velocity).clamp(max=50.0)
            lttc=funcy.mse_loss(ttc_pred,gt_ttc.detach())
        losses={}
        losses.update(lprop)
        losses.update(ldet)
        losses['loss_ttc']=lttc
        if 'depth' in inn[0]:
            gt=torch.stack([x['depth'] for x in inn])
            gt=funcy.interpolate(gt.unsqueeze(1),size=deppre.shape[-2:],mode='nearest')
            valid=gt>0
            losses['ldepth']=funcy.l1_loss(deppre[valid],gt[valid])
        return losses
    def inference(self,inn):
        img=self.preprocess_image(inn)
        feat=self.kamar(img.tensor)
        feat={k: self.att[k](v) for k, v in feat.items()}
        props,_=self.proposal_generator(img,feat,None)
        res,_=self.roi_heads(img,feat,props,None)
        dmap=self.deep(feat)
        return res,dmap

In [8]:
import os, json,cv2
from detectron2.engine import DefaultTrainer, HookBase
from detectron2.config import get_cfg
from detectron2.data import DatasetMapper, build_detection_train_loader, build_detection_test_loader
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator

In [9]:
class Eaarly(HookBase):
    def __init__(self,pat=5,metric='bbox/AP'):
        self.pat=pat
        self.metric=metric
        self.best=-1
        self.count=0
    def after_step(self):
        nexti=self.trainer.iter+1
        if next%self.trainer.cfg.TEST.EVAL_PERIOD==0:
            lmet=self.trainer.storage.latest()
            if self.metric in lmet:
                cs=lmet[self.metric][0]
                if cs>self.best:
                    self.best=cs
                    self.count=0
                    print(f'Nex best {self.metric}:{cs}')
                else:
                    self.count+=1
                    print(f'no improvement in {self.count}')
                if self.count>=self.pat:
                    print('early stop')
                    self.trainer.storage.put_scalar('early_stop',1)
                    raise StopIteration

In [10]:
class Mapppp(DatasetMapper):
    def __init__(self,cfg,ist=True):
        super().__init__(cfg,ist)
        with open('/kaggle/input/filed-chlorseg/filed_data/temporal.json','r') as f:
            self.temporal_map={item['current']:item['previous'] for item in json.load(f)}
    def __call__(self,data):
        data=super().__call__(data)
        cfile=os.path.basename(data['file_name'])
        if cfile in self.temporal_map:
            prev=os.path.join('/kaggle/input/filed-chlorseg/filed_data/image',self.temporal_map[cfile])
            if os.path.exists(prev):
                pimg=cv2.imread(prev)
                pimg=cv2.cvtColor(prev,cv2.COLOR_BGR2RGB)
            data['prev_img']=torch.as_tensor(pimg.transpose(2,0,1))
        return data

In [11]:
class Training(DefaultTrainer):
    @classmethod
    def build_evaluator(cls,cfg,data,out=None):
        return COCOEvaluator(data,cfg,True,out or cfg.OUTPUT_DIR)
    @classmethod
    def build_train_loader(cls,cfg):
        return build_detection_train_loader(cfg,mapper=Mapppp(cfg,True))
    def hooks(self):
        hook=super().hooks()
        hook.append(Eaarly(pat=3))
        return hook

In [12]:
register_coco_instances("cholec_train1", {}, "/kaggle/input/filed-chlorseg/filed_data/train_instances.json", "/kaggle/input/filed-chlorseg/filed_data/images")
register_coco_instances("cholec_val1", {}, "/kaggle/input/filed-chlorseg/filed_data/val_stances.json", "/kaggle/input/filed-chlorseg/filed_data/images")

In [13]:
cfg = get_cfg()

In [14]:
cfg.merge_from_file("./detectron2/configs/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

In [15]:
cfg.MODEL.META_ARCHITECTURE = "TereNaina"

In [16]:
cfg.MODEL.BACKBONE.NAME = "LiHR01"

In [17]:
cfg.DATASETS.TRAIN, cfg.DATASETS.TEST = ("cholec_train1",), ("cholec_val1",)

In [18]:
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 5000
cfg.SOLVER.STEPS=(3000,4500)
cfg.SOLVER.WARMUP_ITERS=500
cfg.TEST.EVAL_PERIOD = 500
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 8
cfg.SOLVER.IMS_PER_BATCH=4

In [19]:
cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5"]
cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256]] 

In [20]:
from detectron2.data import MetadataCatalog

In [21]:
id_map = {5: 0, 9: 1, 13: 2, 14: 3, 15: 4, 16: 5, 17: 6}

MetadataCatalog.get("cholec_train1").set(
    thing_classes=["Grasper", "L-hook", "Trocar", "Scissor", "Clipper", "Irrigator", "SpecimenBag"],
    thing_dataset_id_to_contiguous_id=id_map
)
MetadataCatalog.get("cholec_val1").set(
    thing_classes=["Grasper", "L-hook", "Trocar", "Scissor", "Clipper", "Irrigator", "SpecimenBag"],
    thing_dataset_id_to_contiguous_id=id_map
)

namespace(name='cholec_val1',
          json_file='/kaggle/input/filed-chlorseg/filed_data/val_stances.json',
          image_root='/kaggle/input/filed-chlorseg/filed_data/images',
          evaluator_type='coco',
          thing_classes=['Grasper',
                         'L-hook',
                         'Trocar',
                         'Scissor',
                         'Clipper',
                         'Irrigator',
                         'SpecimenBag'],
          thing_dataset_id_to_contiguous_id={5: 0,
                                             9: 1,
                                             13: 2,
                                             14: 3,
                                             15: 4,
                                             16: 5,
                                             17: 6})

In [22]:
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = Training(cfg)

Model:
TereNaina(
  (kamar): FPN(
    (fpn_lateral2): Conv2d(32, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bottom_up): LiHR01(
      (b1): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (4): 


Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.



Loaded 8080 images in COCO format from /kaggle/input/filed-chlorseg/filed_data/train_instances.json
Removed 3523 images with no usable annotations. 4557 images left.
Distribution of instances among all 7 categories:
|  category   | #instances   |  category  | #instances   |  category  | #instances   |
|:-----------:|:-------------|:----------:|:-------------|:----------:|:-------------|
|   Grasper   | 231          |   L-hook   | 0            |   Trocar   | 27506        |
|   Scissor   | 0            |  Clipper   | 0            | Irrigator  | 0            |
| SpecimenBag | 0            |            |              |            |              |
|    total    | 27737        |            |              |            |              |
Using training sampler TrainingSampler
Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
Serializing 4557 elements to byte tensors and concatenating them all ...
Serialized dataset takes 8.13 MiB
Making batched data loader with

In [23]:
depth_weights = torch.load("hamlyn_depth.pth")
trainer.model.depth_head.load_state_dict(depth_weights, strict=False)
    
trainer.resume_or_load(resume=False)
trainer.train()

FileNotFoundError: [Errno 2] No such file or directory: 'hamlyn_depth.pth'